flowchart TD
  %% Stage 1: Geodata preparation
  subgraph Geodata
    GEC(_archives/scripts/geodata_to_csv.py_)
    GEC -->|writes| GPS_ALL[GPS_ALL.csv]
    GEC -->|writes| GPS_GBS[GPS_GBS_ONLY.csv]
    GBIF(src/py_madaclim/utils/gbif_api.py) -->|reads→writes| RAW_GBIF[GBIF_raw.tsv]
    FMT(format_gbif_data.py) -->|writes| GBIF_FMT[formatted_gbif.csv]
  end

  %% Stage 2: Collection creation
  subgraph Collection
    C1(01_madaclim_collection_creation.ipynb) -->|writes| COLL_ALL[coll_all.csv]
    C1 -->|writes| COLL_BIN[coll_all_bin.csv]
    C1 -->|writes| COLL_CAT[coll_all_categ_nonbin.csv]
  end

  %% Stage 3: Add caffeine class
  subgraph CaffeineClass
    C2(02_add_caff_class_to_collection.ipynb) -->|reads| COLL_ALL  
    C2 -->|writes| CCLASS[coll_caff_node_w_class.csv]
    C2 -->|writes| CCLASS_BIN[coll_caff_node_bin_w_class.csv]
    C2 -->|writes| COORDS[coords_w_caff.csv]
  end

  %% Stage 4: Outlier cleaning
  subgraph Cleaning
    MOUT(03_managing_outliers.ipynb) -->|reads| CCLASS_BIN
    MOUT -->|writes| CLEAN[cleaned_data_w_class.csv]
    MOUT -->|writes| CLEAN_NUM[cleaned_data_num_w_class.csv]
    MOUT -->|writes| CLEAN_CAT[cleaned_data_categ_w_class.csv]
  end

  %% Stage 5: Feature reduction
  subgraph FeatureReduction
    ATTR(04_attribute_analysis.ipynb) -->|reads| CLEAN
    ATTR -->|writes| RED_BIN[reduced_data_bin.csv]
    ARCH(archive/attribute_analysis.ipynb) -->|writes| RED_NUM[reduced_data_num.csv]
  end

  %% Stage 6: Model prep
  subgraph Modeling
    FI(08_feature_importance.ipynb) -->|reads| RED_BIN
    FI -->|writes| TRAIN[reduced_for_training.csv]
    CHM(05_choice_of_model.ipynb) -->|reads| TRAIN
    MREG(07_model_training_regression.ipynb) -->|reads| RED_BIN
    MTEST(09_model_testing.ipynb) -->|reads| TRAIN
  end

  %% Stage 7: Mantel test
  subgraph Mantel
    MNTL(10_mantel_test.ipynb) -->|writes| GEO_DIST[geographic_distances_full.csv]
  end
